# Gradient Descent vs Newton's Method vs Coordinate Descent

When we compute the update direction for a weight — the direction that reduces the loss the most — we assume all other weights stay constant. But they don't. Every weight updates simultaneously.

This notebook explores: what's the difference in convergence when we account for the fact that other weights will also get updated, versus when we don't?

We compare three approaches on $z = x^2 + xy + y^2 + 0.5(x^4 + y^4)$:

1. **Gradient descent** — update all weights simultaneously using partial derivatives. Breaks the assumption.
2. **Coordinate descent** — update only one weight per step (the one with the largest gradient). Never breaks the assumption.
3. **Newton's method** — use the Hessian to account for cross-parameter interactions. The "correct" approach.

Unlike a purely quadratic function, this one has quartic terms — the Hessian changes at every point. Newton's method can't see the full landscape in one step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# z = x^2 + xy + y^2 + 0.5(x^4 + y^4)
def loss(w):
    x, y = w
    return x**2 + x*y + y**2 + 0.5*(x**4 + y**4)

# Gradients — each one depends on BOTH variables
def grad(w):
    x, y = w
    dz_dx = 2*x + y + 2*x**3   # depends on y
    dz_dy = x + 2*y + 2*y**3   # depends on x
    return np.array([dz_dx, dz_dy])

# Hessian — NOT constant, changes with position
def hessian(w):
    x, y = w
    return np.array([[2 + 6*x**2, 1],
                     [1, 2 + 6*y**2]])

# Starting point
w_start = np.array([1.0, 1.0])

print(f"Function: z = x\u00b2 + xy + y\u00b2 + 0.5(x\u2074 + y\u2074)")
print(f"Starting point: {w_start}")
print(f"Starting loss: {loss(w_start)}")
print(f"Minimum: (0, 0), loss = 0")
print(f"")
H = hessian(w_start)
print(f"Hessian at (1, 1):")
print(f"  [{H[0,0]:.0f}  {H[0,1]:.0f}]")
print(f"  [{H[1,0]:.0f}  {H[1,1]:.0f}]")
print(f"")
H0 = hessian(np.array([0.0, 0.0]))
print(f"Hessian at (0, 0):")
print(f"  [{H0[0,0]:.0f}  {H0[0,1]:.0f}]")
print(f"  [{H0[1,0]:.0f}  {H0[1,1]:.0f}]")
print(f"")
print(f"The Hessian changes with position. Newton's method can't solve this in one step.")

## Gradient Descent

Update both weights simultaneously using their partial derivatives. Each `.grad` is computed assuming the other weight is constant — but we update both at once.

In [ ]:
def gradient_descent(w_start, lr, n_steps):
    w = w_start.copy()
    losses = [loss(w)]
    path = [w.copy()]
    
    for step in range(n_steps):
        g = grad(w)
        w = w - lr * g
        losses.append(loss(w))
        path.append(w.copy())
    
    return losses, path

losses_gd, path_gd = gradient_descent(w_start, lr=0.01, n_steps=50)

print(f"Gradient Descent (50 steps, lr=0.01):")
print(f"  Final position: ({path_gd[-1][0]:.6f}, {path_gd[-1][1]:.6f})")
print(f"  Final loss: {losses_gd[-1]:.6f}")

## Coordinate Descent

Update only one weight per step — the one with the largest gradient magnitude. This way, we never break the partial derivative assumption. The other weight really does stay constant.

In [ ]:
def coordinate_descent(w_start, lr, n_steps):
    w = w_start.copy()
    losses = [loss(w)]
    path = [w.copy()]
    
    for step in range(n_steps):
        g = grad(w)
        idx = np.argmax(np.abs(g))
        w[idx] = w[idx] - lr * g[idx]
        losses.append(loss(w))
        path.append(w.copy())
    
    return losses, path

losses_cd, path_cd = coordinate_descent(w_start, lr=0.01, n_steps=50)

print(f"Coordinate Descent (50 steps, lr=0.01):")
print(f"  Final position: ({path_cd[-1][0]:.6f}, {path_cd[-1][1]:.6f})")
print(f"  Final loss: {losses_cd[-1]:.6f}")

## Newton's Method

Use the Hessian to account for cross-parameter interactions. For this function, the Hessian changes at every point — so Newton's method needs multiple steps, not just one.

In [ ]:
def newtons_method(w_start, n_steps):
    w = w_start.copy()
    losses = [loss(w)]
    path = [w.copy()]
    
    for step in range(n_steps):
        g = grad(w)
        H = hessian(w)
        H_inv = np.linalg.inv(H)
        w = w - H_inv @ g
        losses.append(loss(w))
        path.append(w.copy())
    
    return losses, path

losses_newton, path_newton = newtons_method(w_start, n_steps=10)

print(f"Newton's Method (10 steps):")
print(f"  Final position: ({path_newton[-1][0]:.6f}, {path_newton[-1][1]:.6f})")
print(f"  Final loss: {losses_newton[-1]:.6f}")
print(f"")
print(f"Loss at each step:")
for i, l in enumerate(losses_newton):
    print(f"  Step {i}: {l:.6f}")

## The Comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(losses_gd, label='Gradient Descent', linewidth=2)
plt.plot(losses_cd, label='Coordinate Descent', linewidth=2)
plt.plot(range(len(losses_newton)), losses_newton, 'o-', label="Newton's Method", 
         linewidth=2, markersize=8)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('$z = x^2 + xy + y^2 + 0.5(x^4 + y^4)$ \u2014 Three Ways to Optimize')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print(f"After 50 steps:")
print(f"  Gradient Descent:    loss = {losses_gd[-1]:.6f}")
print(f"  Coordinate Descent:  loss = {losses_cd[-1]:.6f}")
print(f"  Newton's Method:     loss = {losses_newton[3]:.6f}  (after just 3 steps)")

## The Takeaway

- **Newton's method** is the fastest — it accounts for cross-parameter interactions via the Hessian and converges in a few steps. But it requires computing and inverting an $N \times N$ Hessian at every step. For a model with millions of parameters, that's impossible.

- **Coordinate descent** never breaks the partial derivative assumption — only one weight changes at a time, so the "hold others constant" condition is genuinely satisfied. But it's painfully slow. With $N$ parameters, you'd need $N$ steps just to touch each weight once.

- **Gradient descent** breaks the assumption — it updates all weights simultaneously using gradients that each assumed the others wouldn't change. But it's $O(N)$ per step, cheap enough to run millions of times.

Gradient descent wins not because it's right, but because it's fast enough to iterate.